In [1]:
import re
# import pandas as pd
from fractions import Fraction
from random import choices
import numpy as np

# NLP Exercises

Exercises from Juravsky & Martin, _Speech and Language Processing_, Third edn., [draft of 12 January 2025](https://web.stanford.edu/~jurafsky/slp3/old_jan25/).

## Part I: Fundamental Algorithms for NLP

### Chapter 3: N-gram Language Models

**3.1** Write out the equation for trigram probability estimation (modifying Eq. 3.11).

- $$P(w_n|w_{n-2}w_{n-1})=\frac{C(w_{n-2}w_{n-1}w_n)}{C(w_{n-2}w_{n-1})}$$

Now write out all the non-zero trigram probabilities for the _I am Sam_ corpus on page 35.

- \begin{gather*}
P(\textrm{I}|\langle s\rangle\langle s\rangle)=\frac{2}{3}\qquad P(\textrm{am}|\langle s\rangle\textrm{ I})=\frac{1}{2}\qquad P(\textrm{Sam}|\textrm{I am})=\frac{1}{2}\qquad P(\langle/s\rangle|\textrm{am Sam})=1\qquad P(\textrm{Sam}|\langle s\rangle\langle s\rangle)=\frac{1}{3}\\
P(\textrm{I}|\langle s\rangle\textrm{ Sam})=1\qquad P(\textrm{am}|\textrm{Sam I})=1\qquad P(\langle/s\rangle|\textrm{I am})=\frac{1}{2}\qquad P(\textrm{do}|\langle s\rangle\textrm{ I})=\frac{1}{2}\\
P(\textrm{not}|\textrm{I do})=P(\textrm{like}|\textrm{do not})=P(\textrm{green}|\textrm{not like})=P(\textrm{eggs}|\textrm{like green})=P(\textrm{and}|\textrm{green eggs})=P(\textrm{ham}|\textrm{eggs and})=P(\langle/s\rangle|\textrm{and ham})=1
\end{gather*}

**3.2** Calculate the probability of the sentence _i want chinese food_. Give two probabilities, one using Fig. 3.2 and the ‘useful probabilities’ just below it on page 37, and another using the add-1 smoothed table in Fig. 3.7. Assume the additional add-1 smoothed probabilities $P(\textrm{i}|\langle s\rangle)=0.19$ and $P(\langle/s\rangle|\textrm{food})=0.40$.

- \begin{align*}
P(\langle s\rangle\textrm{ i want chinese food }\langle/s\rangle)&=P(\textrm{i}|\langle s\rangle)\times P(\textrm{want}|\textrm{i})\times P(\textrm{chinese}|\textrm{want})\times P(\textrm{food}|\textrm{chinese})\times P(\langle/s\rangle|\textrm{food})\\
&=0.25\times 0.33\times 0.0065\times 0.52\times 0.68
\end{align*}

In [2]:
print("=",0.25*0.33*0.0065*0.52*0.68)

= 0.00018961800000000004


With add-1 smoothing:

$$=0.19\times 0.21\times 0.0029\times 0.0052\times 0.4$$

In [3]:
print("=",0.19*0.21*0.0029*0.0052*0.4)

= 2.4067679999999994e-07


**3.3** Which of the two probabilities you computed in the previous exercise is higher, unsmoothed or smoothed? Explain why.

- The unsmoothed probability is approximately 1000 times higher. The reason is that in the smoothed case, probabilities are taken away from each of these bigrams to add to non-existent bigrams so that the total probability adds to 1 without any zero probabilities.

**3.4** We are given the following corpus, modified from the one in the chapter:

\begin{gather*}
\langle s\rangle\textrm{ I am Sam }\langle/s\rangle\\
\langle s\rangle\textrm{ Sam I am }\langle/s\rangle\\
\langle s\rangle\textrm{ I am Sam }\langle/s\rangle\\
\langle s\rangle\textrm{ I do not like green eggs and Sam }\langle/s\rangle\\
\end{gather*}

Using a bigram language model with add-one smoothing, what is $P(\textrm{Sam}|\textrm{am})$? Include $\langle s\rangle$ and $\langle/s\rangle$ in your counts just like any other token.

- Bigram counts with add-1 smoothing:

|  | $\langle s\rangle$ | i | am | sam | $\langle/s\rangle$ | do | not | like | green | eggs | and |
| - | - | - | - | - | - | - | - | - | - | - | - |
| $\langle s\rangle$ | 1 | 1 | 2 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |
| i | 1 | 1 | 4 | 1 | 1 | 2 | 1 | 1 | 1 | 1 | 1 |
| am | 1 | 1 | 1 | 3 | 2 | 1 | 1 | 1 | 1 | 1 | 1 |
| sam | 1 | 2 | 1 | 1 | 3 | 1 | 1 | 1 | 1 | 1 | 1 |
| $\langle/s\rangle$ | 5 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |
| do | 1 | 1 | 1 | 1 | 1 | 1 | 2 | 1 | 1 | 1 | 1 |
| not | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 2 | 1 | 1 | 1 |
| like | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 2 | 1 | 1 |
| green | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 2 | 1 |
| eggs | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 2 |
| and | 1 | 1 | 1 | 2 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |

- According to the above table, $P(\textrm{Sam}|\textrm{am})=\frac{3}{14}$.

**3.5** Suppose we didn’t use the end-symbol $\langle/s\rangle$. Train an unsmoothed bigram grammar on the following training corpus without using the end-symbol $\langle/s\rangle$:

\begin{gather*}
\langle s\rangle\textrm{ a b}\\
\langle s\rangle\textrm{ b b}\\
\langle s\rangle\textrm{ b a}\\
\langle s\rangle\textrm{ a a}\\
\end{gather*}

Demonstrate that your bigram model does not assign a single probability distribution across all sentence lengths by showing that the sum of the probability of the four possible 2 word sentences over the alphabet $\{a,b\}$ is 1.0, and the sum of the probability of all possible 3 word sentences over the alphabet $\{a,b\}$ is also 1.0.

- Bigram counts:

| | $\langle s\rangle$ | a | b |
| - | - | - | - |
| $\langle s\rangle$ | 0 | 2 | 2 |
| a | 0 | 1 | 1 |
| b | 0 | 1 | 1 |

- Two-word sentences and their probabilities:
    - $P(\langle s\rangle\textrm{ a a})=\frac{1}{2}\times\frac{1}{2}=\frac{1}{4}$
    - $P(\langle s\rangle\textrm{ a b})=\frac{1}{2}\times\frac{1}{2}=\frac{1}{4}$
    - $P(\langle s\rangle\textrm{ b a})=\frac{1}{2}\times\frac{1}{2}=\frac{1}{4}$
    - $P(\langle s\rangle\textrm{ b b})=\frac{1}{2}\times\frac{1}{2}=\frac{1}{4}$
    - Sum: 1.
- Three-word sentences and their probabilities:
    - $P(\langle s\rangle\textrm{ a a a})=\frac{1}{2}\times\frac{1}{2}\times\frac{1}{2}=\frac{1}{8}$
    - $P(\langle s\rangle\textrm{ a a b})=\frac{1}{2}\times\frac{1}{2}\times\frac{1}{2}=\frac{1}{8}$
    - $P(\langle s\rangle\textrm{ a b a})=\frac{1}{2}\times\frac{1}{2}\times\frac{1}{2}=\frac{1}{8}$
    - $P(\langle s\rangle\textrm{ a b b})=\frac{1}{2}\times\frac{1}{2}\times\frac{1}{2}=\frac{1}{8}$
    - $P(\langle s\rangle\textrm{ b a a})=\frac{1}{2}\times\frac{1}{2}\times\frac{1}{2}=\frac{1}{8}$
    - $P(\langle s\rangle\textrm{ b a b})=\frac{1}{2}\times\frac{1}{2}\times\frac{1}{2}=\frac{1}{8}$
    - $P(\langle s\rangle\textrm{ b b a})=\frac{1}{2}\times\frac{1}{2}\times\frac{1}{2}=\frac{1}{8}$
    - $P(\langle s\rangle\textrm{ b b b})=\frac{1}{2}\times\frac{1}{2}\times\frac{1}{2}=\frac{1}{8}$
    - Sum: 1.

**3.6** Suppose we train a trigram language model with add-one smoothing on a given corpus. The corpus contains V word types. Express a formula for estimating $P(w_3|w_1,w_2)$, where $w_3$ is a word which follows the bigram $(w_1,w_2)$, in terms of various n-gram counts and V. Use the notation $c(w_1,w_2,w_3)$ to denote the number of times that trigram $(w_1,w_2,w_3)$ occurs in the corpus, and so on for bigrams and unigrams.

- $$P(w_3|w_1,w_2)=\frac{c(w_1,w_2,w_3)}{c(w_1,w_2)}$$

**3.7** We are given the following corpus, modified from the one in the chapter:

\begin{gather*}
\langle s\rangle\textrm{ I am Sam }\langle/s\rangle\\
\langle s\rangle\textrm{ Sam I am }\langle/s\rangle\\
\langle s\rangle\textrm{ I am Sam }\langle/s\rangle\\
\langle s\rangle\textrm{ I do not like green eggs and Sam }\langle/s\rangle\\
\end{gather*}

If we use linear interpolation smoothing between a maximum-likelihood bigram model and a maximum-likelihood unigram model with $\lambda_1=\frac{1}{2}$ and $\lambda_2=\frac{1}{2}$, what is $P(\textrm{Sam}|\textrm{am})$? Include $\langle s\rangle$ and $\langle/s\rangle$ in your counts just like any other token.

- \begin{align*}\hat{P}(\textrm{Sam}|\textrm{am})&=\lambda_1 P(\textrm{Sam})+\lambda_2 P(\textrm{Sam}|\textrm{am})\\
&=\frac{1}{2}\times\frac{4}{17}+\frac{1}{2}\times\frac{2}{3}
\end{align*}

<!-- Unsmoothed bigram counts (counts from the answer to **3.4** above, reduced by 1 in each case):

|  | $\langle s\rangle$ | i | am | sam | $\langle/s\rangle$ | do | not | like | green | eggs | and |
| - | - | - | - | - | - | - | - | - | - | - | - |
| $\langle s\rangle$ | 0 | 0 | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| i | 0 | 0 | 3 | 0 | 0 | 1 | 0 | 0 | 0 | 0 | 0 |
| am | 0 | 0 | 0 | 2 | 1 | 0 | 0 | 0 | 0 | 0 | 0 |
| sam | 0 | 1 | 0 | 0 | 2 | 0 | 0 | 0 | 0 | 0 | 0 |
| $\langle/s\rangle$ | 4 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| do | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 0 | 0 | 0 | 0 |
| not | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 2 | 1 | 1 | 1 |
| like | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 2 | 1 | 1 |
| green | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 2 | 1 |
| eggs | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 1 | 2 |
| and | 1 | 1 | 1 | 2 | 1 | 1 | 1 | 1 | 1 | 1 | 1 |-->

In [4]:
print("=",Fraction(1,2)*Fraction(4,17)+Fraction(1,2)*Fraction(2,3))

= 23/51


**3.8** Write a program to compute unsmoothed unigrams and bigrams.

In [5]:
def unigram_count(word:str, text:str) -> Fraction:
    word = word.lower()
    text_list = [w.strip(",\"'.!?") for w in text.split()]
    numerator = len([w for w in text_list if w.lower()==word])
    denominator = len(text_list)
    return Fraction(numerator,denominator)

def bigram_count(word2:str, word1:str, text:str) -> Fraction:
    word2, word1 = (w.lower() for w in [word2, word1])
    sentences = [f"<s> {sentence} </s>" for sentence in re.split(r'[\.!?]', text)]
    numerator = 0
    denominator = 0
    for sentence in sentences:
        for i in range(len(sentence.split())-1):
            j = i+1
            if sentence.split()[i].strip(",\"'").lower()==word1:
                denominator += 1
                if sentence.split()[j].strip(",\"'").lower()==word2:
                    numerator += 1
    return Fraction(numerator,denominator)

In [6]:
sam_text = "I am Sam. Sam I am. I am Sam. I do not like green eggs and Sam."

In [7]:
print(unigram_count('Sam', sam_text))

4/17


In [8]:
print(bigram_count("Sam", "am", sam_text))

2/3


In [9]:
# This is too slow in practice, because of the repeated looping.
# def grams(text:str, include_zeroes:bool=False) -> dict:
#     words = {w.strip(",\"'.!?") for w in text.split()}
#     output = {}
#     # unigrams
#     for word in words:
#         output[word.lower()] = unigram_count(word, text)
#     # bigrams
#     for word0 in words:
#         for word1 in words.difference({word0}):
#             output[f"{word0.lower()} {word1.lower()}"] = bigram_count(word1, word0, text)
#     if include_zeroes:
#         return output
#     return {k:v for k,v in output.items() if v>0}

def grams(text:str) -> dict:
    output = {'unigrams':{}, 'bigrams':{}}
    # unigrams
    tokens = [w.strip(";,\"'.!?").lower() for w in text.split()]
    types = set(tokens)
    for word in types:
        output['unigrams'][word] = Fraction(tokens.count(word),len(tokens))
    # bigrams
    sentences = [f"<s> {sentence} </s>" for sentence in re.split(r'[\.!?]', text)
                 if len(sentence)>0]
    for sentence in sentences:
        words = [w.strip(";,\"'").lower() for w in sentence.split()]
        for i in range(len(words)-1):
            bigram = " ".join(words[i:i+2])
            denominator = tokens.count(words[i]) if i>0 else len(sentences) #<s> at the start of each sentence
            if bigram not in output['bigrams'].keys():
                output['bigrams'][bigram] = Fraction(1,denominator)
            else:
                output['bigrams'][bigram] += Fraction(1,denominator)
    return output

In [10]:
both_grams = grams(sam_text)
print('Unigrams\n--------')
for k,v in both_grams['unigrams'].items():
    print(f"{k}\t{v}")
print('\nBigrams\n-------')
for k,v in both_grams['bigrams'].items():
    print(f"{k}{"\t"*(1-int(len(k)/8)+1)}{v}")

Unigrams
--------
i	4/17
like	1/17
green	1/17
do	1/17
and	1/17
sam	4/17
am	3/17
not	1/17
eggs	1/17

Bigrams
-------
<s> i		3/4
i am		3/4
am sam		2/3
sam </s>	3/4
<s> sam		1/4
sam i		1/4
am </s>		1/3
i do		1/4
do not		1
not like	1
like green	1
green eggs	1
eggs and	1
and sam		1


**3.9** Run your n-gram program on two different small corpora of your choice (you might use email text or newsgroups). Now compare the statistics of the two corpora. What are the differences in the most common unigrams between the two? How about interesting differences in bigrams?

In [15]:
# Universal Declaration of Human Rights
with open('UDHR.txt', 'r') as udhr_file:
    udhr = udhr_file.read().replace('\n', ' ')

In [16]:
print("""Universal Declaration of Human Rights
-------------------------------------
""")
print('Unigrams\n--------')
for k,v in sorted([kv for kv in grams(udhr)['unigrams'].items()], 
                  key=lambda x:x[1], reverse=True):
    print(f"{k}{"\t"*(2-int(len(k)/8)+1)}{v}")
print('\nBigrams\n-------')
for k,v in sorted([kv for kv in grams(udhr)['bigrams'].items()],
                  key=lambda x:x[1], reverse=True):
    print(f"{k}{"\t"*(3-int(len(k)/8)+1)}{v}")

Universal Declaration of Human Rights
-------------------------------------

Unigrams
--------
the			120/1741
and			106/1741
of			90/1741
to			83/1741
in			43/1741
right			33/1741
be			31/1741
everyone		30/1741
or			30/1741
article			30/1741
has			28/1741
shall			27/1741
rights			21/1741
his			21/1741
a			19/1741
any			18/1741
for			17/1741
is			13/1741
by			13/1741
human			12/1741
all			12/1741
equal			11/1741
as			11/1741
this			11/1741
freedom			11/1741
freedoms		10/1741
no			10/1741
one			10/1741
law			9/1741
which			9/1741
entitled		9/1741
protection		9/1741
with			9/1741
nations			8/1741
have			8/1741
education		8/1741
are			8/1741
social			8/1741
free			7/1741
whereas			7/1741
family			6/1741
fundamental		6/1741
full			6/1741
other			6/1741
at			6/1741
against			6/1741
declaration		6/1741
person			5/1741
national		5/1741
public			5/1741
society			5/1741
dignity			5/1741
from			5/1741
religion		5/1741
country			5/1741
international		5/1741
their			5/1741
without			5/1741
it			5/1

In [17]:
# 1 John
with open('1_John.txt', 'r') as first_john_file:
    first_john = first_john_file.read().replace('\n', ' ')

In [18]:
print("""The First Epistle of John
-------------------------
""")
print('Unigrams\n--------')
for k,v in sorted([kv for kv in grams(first_john)['unigrams'].items()], 
                  key=lambda x:x[1], reverse=True):
    print(f"{k}{"\t"*(2-int(len(k)/8)+1)}{v}")
print('\nBigrams\n-------')
for k,v in sorted([kv for kv in grams(first_john)['bigrams'].items()],
                  key=lambda x:x[1], reverse=True):
    print(f"{k}{"\t"*(3-int(len(k)/8)+1)}{v}")

The First Epistle of John
-------------------------

Unigrams
--------
the			151/2517
that			124/2517
and			40/839
is			98/2517
of			82/2517
we			80/2517
in			73/2517
he			24/839
god			19/839
not			55/2517
him			52/2517
have			43/2517
his			43/2517
us			11/839
love			11/839
ye			32/2517
know			9/839
for			26/2517
hath			26/2517
you			26/2517
because			25/2517
son			23/2517
but			23/2517
unto			22/2517
if			7/839
this			7/839
are			7/839
world			20/2517
from			17/2517
which			16/2517
one			16/2517
i			5/839
sin			5/839
our			5/839
life			5/839
shall			14/2517
children		4/839
jesus			4/839
father			4/839
brother			4/839
spirit			4/839
as			11/2517
no			11/2517
a			11/2517
even			11/2517
it			11/2517
to			11/2517
they			10/2517
christ			10/2517
whosoever		10/2517
also			3/839
beginning		3/839
truth			3/839
heard			3/839
with			3/839
things			3/839
all			3/839
loveth			3/839
little			3/839
man			8/2517
write			8/2517
hereby			8/2517
was			8/2517
seen			8/2517
abideth			8/2517
word			7/2517

#### Unigrams

- | | UDHR | 1 John |
| - | - | - |
| 1 | the | the |
| 2 | and | is |
| 3 | of | and |
| 4 | to | we |
| 5 | in | in |
| 6 | right | you |
| 7 | be | god |
| 8 | or | that |
| 9 | everyone | to |
| 10 | article | not |

- The top 3 don't tell you much, but by the time you get down to numbers 4 and 6 in 1 John you've got some strong clues about what kind of document it is&mdash;a letter&mdash;and then "God" being number 7 shows that it's a religious text. For the UDHR, "right" at number 6 is the first clue as to the document type, and then "article" at number 10 shows it more clearly.

#### Bigrams

- The highest probability bigrams have probability 1, almost always because the first word in the bigram is rare. It's more interesing to look at the highest probability with probability less than 1.

**3.10** Add an option to your program to generate random sentences.

In [19]:
# I'll take this as an instruction to generate random sentences based on a bigram
def random_sentence(bigrams:dict) -> str:
    output = ["<s>"]
    finished = False
    while not finished:
        candidates = [(k,v) for k,v in bigrams.items() if k.split()[0]==output[-1]]
        selection = choices([c[0] for c in candidates],
                            weights=[c[1] for c in candidates])[0].split()[-1]
        output.append(selection)
        if output[-1]=='</s>':
            finished = True
    return " ".join(output[1:-1])+'.'

In [20]:
random_sentence(grams(udhr)['bigrams'])

'parents have a world in the protection against any other means of sovereignty.'

In [22]:
random_sentence(grams(first_john)['bigrams'])

'little children of god.'

**3.11** Add an option to your program to compute the perplexity of a test set.

In [23]:
# I'll write separate functions for unigram and bigram model perplexity.
def unigram_perplexity(test_set:str, unigrams:dict) -> float:
    tokens = [w.strip(";,\"'.!?").lower() for w in test_set.split()]
    num_words = len(tokens)
    # log_denom = 0
    # for token in tokens:
    #     try:
            # log_denom += np.log(unigrams[token])
        # except TypeError:
        #     print(token)
    log_denom = sum([np.log(float(unigrams[token])) for token in tokens])
    return (1/np.exp(log_denom))**(1/num_words)

In [24]:
# The probabilities come out underflowingly small for the longer texts.
unigram_perplexity(sam_text, grams(sam_text)['unigrams'])

np.float64(7.293377686774777)

In [25]:
def bigram_perplexity(test_set:str, bigrams:dict) -> float:
    sentences = [f"<s> {sentence} </s>" for sentence in re.split(r'[\.!?]', test_set)
                 if len(sentence)>0]
    pairs = []
    for sentence in sentences:
        words = [w.strip(";,\"'").lower() for w in sentence.split()]
        for i in range(len(words)-1):
            pairs.append(" ".join(words[i:i+2]))
    log_denom = sum([np.log(float(bigrams[pair])) for pair in pairs])
    num_bigrams = len(pairs)
    return (1/np.exp(log_denom))**(1/num_bigrams)

In [26]:
bigram_perplexity(sam_text, grams(sam_text)['bigrams'])

np.float64(1.5102345408393125)

**3.12** You are given a training set of 100 numbers that consists of 91 zeros and 1 each of the other digits 1-9. Now we see the following test set: 0 0 0 0 0 3 0 0 0 0. What is the unigram perplexity?

- By hand: \begin{align*}
\textrm{perplexity}(W)&=\sqrt[N]{\prod_{i=1}^{N}\frac{1}{P(w_i)}}\\
&=\sqrt[10]{\frac{1}{P(0)^9\times P(3)}}\\
&=\sqrt[10]{\frac{1}{0.91^9\times 0.01}}\\
&=\sqrt[10]{233.6831882}\\
&=1.72529255
\end{align*}
- Checking my functions:

In [27]:
num_train = "1 2 3 4 5 6 7 8 9"+" 0"*91
num_test = "0 0 0 0 0 3 0 0 0 0"
print("Training set:",num_train)
print("\nTest set:",num_test)
print("\nPerplexity of the test set:",unigram_perplexity(num_test,grams(num_train)['unigrams']))

Training set: 1 2 3 4 5 6 7 8 9 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

Test set: 0 0 0 0 0 3 0 0 0 0

Perplexity of the test set: 1.7252925496828493
